# RQ2 pairwise external-validity gates — CPU only

Runs Gate A (width vs model-profiled log-FLOPs), Gate B0 (LOEO + LPO), freezes both predictors, and optionally evaluates Gate B1 if a disjoint fresh-state artifact is attached. This notebook never trains a network and never refits on fresh states.

In [ ]:
import os, subprocess, sys, json, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)

## Resolve the six-state dynamic-geometry diagnostic

In [ ]:
import importlib
import rq2_pairwise_external_gates, rq2_pairwise_surrogate_regret, rq2_quick_trajectory_diagnostic
rq2_pairwise_external_gates = importlib.reload(rq2_pairwise_external_gates)
rq2_pairwise_surrogate_regret = importlib.reload(rq2_pairwise_surrogate_regret)
rq2_quick_trajectory_diagnostic = importlib.reload(rq2_quick_trajectory_diagnostic)
QUICK_ROOT = rq2_pairwise_surrogate_regret.find_quick_trajectory_root(
    Path('/kaggle/input'), '/kaggle/working/materialized-pairwise-gates-quick'
)
try:
    HT_ROOT = rq2_quick_trajectory_diagnostic.find_ht_development_root(
        Path('/kaggle/input'), '/kaggle/working/materialized-pairwise-gates-ht'
    )
except FileNotFoundError:
    HT_ROOT = None
print('Quick root:', QUICK_ROOT)
print('Optional HT FLOPs fallback:', HT_ROOT)

## Gate A + Gate B0, then freeze predictors before fresh states

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/rq2-pairwise-external-gates')
started = time.perf_counter()
development = rq2_pairwise_external_gates.run_all_development_gates(
    QUICK_ROOT, OUTPUT_DIR, ht_root=HT_ROOT
)
development['git_commit'] = GIT_COMMIT
(OUTPUT_DIR/'gate_a_b0_complete.json').write_text(json.dumps(development, indent=2)+'\n')
print(f'Gate A+B0 completed in {time.perf_counter()-started:.1f}s')
print(json.dumps(development, indent=2))

## Inspect Gate A and held-out Gate B0

In [ ]:
import pandas as pd
from IPython.display import display, Image
print('GATE A'); display(pd.DataFrame([json.loads((OUTPUT_DIR/'gate_a_summary.json').read_text())]).drop(columns=['flops']))
print('GATE B0'); display(pd.read_csv(OUTPUT_DIR/'gate_b0_summary.csv'))
display(pd.read_csv(OUTPUT_DIR/'gate_b0_heldout_exact_variance.csv'))
display(Image(filename=str(OUTPUT_DIR/'gate_b0_exact_variance_delta.png')))

## Optional Gate B1 — disjoint fresh states only

If an attached dataset contains `fresh_state_metadata.json`, `fresh_pair_structure.csv`, and `grams/*.npy`, this cell evaluates it with the frozen predictor. It never fits on those states. Without that artifact the notebook correctly remains at `FRESH_GATE_PENDING`.

In [ ]:
fresh_metadata = sorted(Path('/kaggle/input').rglob('fresh_state_metadata.json'))
if len(fresh_metadata) == 0:
    print('Gate B1 pending: attach one fresh-state artifact. Frozen predictor:', OUTPUT_DIR/'frozen_pairwise_predictors.json')
    fresh_summary = None
else:
    assert len(fresh_metadata) == 1, f'Expected one fresh-state artifact, found {fresh_metadata}'
    FRESH_ROOT = fresh_metadata[0].parent
    fresh_summary = rq2_pairwise_external_gates.run_gate_b1_fresh(
        FRESH_ROOT, OUTPUT_DIR/'frozen_pairwise_predictors.json', OUTPUT_DIR/'fresh_gate_b1'
    )
    print(json.dumps(fresh_summary, indent=2))
    display(pd.read_csv(OUTPUT_DIR/'fresh_gate_b1/gate_b1_fresh_exact_variance.csv'))

## Validate and export

In [ ]:
required = [
 'gate_a_resource_pair_scores.csv','gate_a_resource_pair_policies.csv',
 'gate_a_resource_exact_variance.csv','gate_a_summary.json',
 'gate_b0_heldout_predictions.csv','gate_b0_heldout_metrics.csv',
 'gate_b0_heldout_exact_variance.csv','gate_b0_summary.csv',
 'frozen_pairwise_predictors.json','gate_a_b0_complete.json'
]
missing = [name for name in required if not (OUTPUT_DIR/name).is_file() or (OUTPUT_DIR/name).stat().st_size == 0]
assert not missing, f'Missing gate artifacts: {missing}'
frozen = json.loads((OUTPUT_DIR/'frozen_pairwise_predictors.json').read_text())
assert frozen['status'] == 'FROZEN_BEFORE_FRESH_STATES' and frozen['no_refit_on_fresh_states'] is True
bundle_path = Path('/kaggle/working/rq2-pairwise-external-gates.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file(): bundle.write(path, path.relative_to(OUTPUT_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**20:.1f} MiB')
bundle_path